# Multiscale quickstart

This notebook makes a small synthetic recording, then computes temporal, spectral, and time-changing DD. Replace `signals` with your own preprocessed `(nodes, samples[, trials])` array. The short optimisation settings are for learning the API; increase restarts and iteration limits for research results.


## Your data and preprocessing

Keep the same node order and preprocessing in every subject, condition, and window. For EEG, record the reference, filtering, artefacts, sampling rate, and source reconstruction. For fMRI, record parcellation, nuisance handling, filtering, TR, and any deconvolution. Windows are sample ranges, so convert physical time using your sampling rate.


In [ ]:
import numpy as np

rng = np.random.default_rng(8)
signals = np.zeros((3, 600))
A_true = np.array([[0.6, 0.1, 0.0], [0.0, 0.5, 0.1], [0.0, 0.0, 0.4]])
for t in range(1, signals.shape[1]):
    signals[:, t] = A_true @ signals[:, t-1] + rng.normal(size=3)
fs = 64  # samples per second
# For your own data: signals = np.load("path/to/preprocessed_signals.npy")


## Fit a VAR model and search the full recording

Choose the maximum model order and criterion for your signal. The same model supports temporal and spectral DD.


In [ ]:
from multiscale import fit_var_auto, run_grid_parallel, random_initial_bases, run_full_dd

fit = fit_var_auto(signals, pmax=5, criterion="aic", regmode="LWR")
A, V = fit.fit.coefficients, fit.fit.covariance
print("Selected order:", fit.order)

spectral = run_grid_parallel(A, V, fs=fs, fres=32, dimensions=[1],
                             bands={"slow": (2, 12)}, runs=4, workers=1,
                             pre_iters=20, final_iters=20, seed=7)
starts = random_initial_bases(3, 1, runs=4, seed=7)
temporal = run_full_dd(A, V, fs, 32, starts, final_method="state_space",
                       pre_iters=20, state_space_iters=10, seed=7)
print("Spectral DD:", spectral[(1, "slow")]["final"].values[0])
print("Temporal DD:", temporal["state_space"].values[0])


## Time-changing DD

Each window gets a separate model fit and optimisation. Here the windows do not overlap, but they may. The same function also accepts `domain="temporal"` without bands.


In [ ]:
from multiscale import run_time_changing_dd, minimum_dd_by_window

windows = {"early": (0, 300), "late": (300, 600)}
trajectory = run_time_changing_dd(signals, windows, domain="spectral",
                                  fs=fs, fres=32, dimensions=[1],
                                  bands={"slow": (2, 12)}, pmax=5,
                                  criterion="aic", runs=4, workers=1,
                                  pre_iters=20, final_iters=20, seed=7)
minimum_dd_by_window(trajectory, domain="spectral")


## Optional figure

Install `./multiscale[plots]` first. The figure is saved to your chosen output path and is not part of the package.


In [ ]:
from multiscale.analysis import summarise_window_results
from multiscale.visualisation import plot_time_changing_dd

window_table = summarise_window_results(trajectory, domain="spectral")
plot_time_changing_dd(window_table, "figures/time_changing_dd")
